# Multimodal Fusion: DaTSCAN CNN + Clinical Tabular Features

This notebook implements and compares three fusion strategies for combining your best pretrained CNN with clinical tabular data from PPMI.

## Structure
1. **Config** — set your paths and feature groups here
2. **Data loading** — merge image paths with tabular features
3. **Late fusion** — combine existing CNN output probabilities with tabular-only ML
4. **Feature fusion** — strip CNN head, concatenate embeddings with tabular branch
5. **Information gain table** — compare all combinations
6. **Visualisation** — boxplots

**Before running:** make sure your best CNN weights `.pth` file is available and your PPMI excel/csv is loaded.

## 0. Imports

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from monai.data import Dataset as MonaiDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, recall_score, precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('..'))
from src.architectures import ParkinsonClassifier2D, ParkinsonClassifier25D
from src.transforms import get_2d_sum_transforms_padding, get_25d_transforms_padding

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Config

Set your file paths and choose which tabular feature groups to experiment with.

In [ ]:
# Paths 
IMAGE_CSV    = 'data/ppmi_rawdata_sesBL_mapping.csv'   # your raw image mapping
TABULAR_CSV  = 'data/ppmi_tabular.csv'                 # your PPMI excel exported as csv

# Best CNN weights — change this to whichever run you want to use as backbone
# e.g. your 25d_resnet_raw run which had AUC ~0.982
CNN_WEIGHTS  = '../outputs/25d_resnet_raw_2fold_XXXXXXXX/fold_1/best_model.pth'
CNN_CLASS    = ParkinsonClassifier25D   # must match the weights above
CNN_EMBED_DIM = 512                     # 512 for 25d_resnet, 32 for 2D/3D custom CNNs

# Transform must match what was used when training the CNN above
ROI_SIZE    = (76, 76, 76)
TRANSFORM   = get_25d_transforms_padding(ROI_SIZE)

# ── Training hyperparams ─────────────────────────────────────────────────
FOLDS       = 2        # keep at 2 for fast screening, increase to 5 for final
EPOCHS      = 50       # fusion head trains faster than a full CNN
LR          = 1e-4
BATCH_SIZE  = 8
DROPOUT     = 0.3
SEED        = 42
FREEZE_CNN  = True     # True = only train the fusion head (faster, less overfitting)
                       # False = fine-tune the whole network end-to-end

# ── Tabular feature groups 
# Edit column names to match your actual PPMI excel headers.
# Each group is tested independently AND all combined.
FEATURE_GROUPS = {
    'smell_upsit':   ['UPSIT_TOTAL'],
    'motor_updrs':   ['UPDRS3_TOTAL'],          # UPDRS Part III (motor)
    'demographics':  ['AGE', 'SEX'],
    'cognitive':     ['MOCA_TOTAL'],
}

# Patient ID column — used to join image CSV and tabular CSV
PATIENT_ID_COL = 'PATNO'

print('Config loaded.')
print(f'Feature groups: {list(FEATURE_GROUPS.keys())}')

## 2. Load and merge data

Join your image mapping CSV with the tabular PPMI data on patient ID.

In [ ]:
img_df = pd.read_csv(IMAGE_CSV)
tab_df = pd.read_csv(TABULAR_CSV)

print(f'Image CSV:    {len(img_df)} rows, columns: {list(img_df.columns)}')
print(f'Tabular CSV:  {len(tab_df)} rows')
print(f'\nTabular CSV columns (first 20): {list(tab_df.columns[:20])}')

In [ ]:
# ── Merge on patient ID ───────────────────────────────────────────────────
# Keep only baseline visit tabular data if there are multiple timepoints
# Uncomment and adjust if your tabular CSV has a visit column:
# tab_df = tab_df[tab_df['VISIT'] == 'BL']

# Collect all feature columns we need
all_features = [f for group in FEATURE_GROUPS.values() for f in group]
tab_subset   = tab_df[[PATIENT_ID_COL] + all_features].drop_duplicates(subset=PATIENT_ID_COL)

merged = img_df.merge(tab_subset, on=PATIENT_ID_COL, how='inner')
print(f'After merge: {len(merged)} patients with both image and tabular data')
print(f'Missing values per feature:')
print(merged[all_features].isnull().sum())

In [ ]:
# ── Drop rows with missing values in any feature group ────────────────────
# Note: in your thesis you should discuss this — imputation is an alternative
merged_clean = merged.dropna(subset=all_features).reset_index(drop=True)
print(f'After dropping NaN rows: {len(merged_clean)} patients')

# ── Balance: downsample PD to match HC ───────────────────────────────────
hc_df = merged_clean[merged_clean['label'] == 0]
pd_df = merged_clean[merged_clean['label'] == 1].sample(n=len(hc_df), random_state=SEED)
balanced = pd.concat([hc_df, pd_df]).reset_index(drop=True)
print(f'Balanced: {len(balanced)} total ({len(hc_df)} HC, {len(hc_df)} PD)')
balanced.head()

## 3. Helper: metrics for one fold

In [ ]:
def compute_metrics(labels, probs):
    preds = (probs > 0.5).astype(int)
    return {
        'auc':       roc_auc_score(labels, probs),
        'f1':        f1_score(labels, preds, zero_division=0),
        'accuracy':  accuracy_score(labels, preds),
        'recall':    recall_score(labels, preds, zero_division=0),
        'precision': precision_score(labels, preds, zero_division=0),
    }

## 4. Late fusion (baseline)

No new training needed:
- Your CNN already outputs a probability for each patient
- Train a simple LR/SVM on tabular features
- Average both probability outputs

This is a quick sanity check. If feature fusion later doesn't beat this, something is wrong with the fusion architecture.

In [ ]:
def get_cnn_probabilities(df, model_class, weights_path, transform, embed_dim, device, extract_embedding=False):
    """
    Run inference with the full CNN and return probabilities (or embeddings).
    
    extract_embedding=False → returns sigmoid probs (B,)
    extract_embedding=True  → strips last layer, returns embeddings (B, embed_dim)
    """
    model = model_class(dropout_rate=DROPOUT)
    model.load_state_dict(torch.load(weights_path, map_location=device))
    
    if extract_embedding:
        # Strip the final classification layer
        if hasattr(model, 'fc2'):   # custom 2D/3D CNNs
            model.fc2 = nn.Identity()
        elif hasattr(model, 'fc'):  # 25d_resnet
            model.fc = nn.Identity()
    
    model = model.to(device)
    model.eval()
    
    files = [{'image': p, 'label': l} for p, l in zip(df['path'], df['label'])]
    ds    = MonaiDataset(data=files, transform=transform)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
    
    all_out, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            imgs   = batch['image'].to(device)
            labels = batch['label'].numpy().flatten()
            out    = model(imgs).cpu().numpy()
            if not extract_embedding:
                out = 1 / (1 + np.exp(-out.flatten()))  # sigmoid
            all_out.extend(out)
            all_labels.extend(labels)
    
    return np.array(all_out), np.array(all_labels)

In [ ]:
def run_late_fusion_cv(df, feature_cols, n_folds=FOLDS, alpha=0.5):
    """
    Late fusion: average CNN probabilities with tabular LR probabilities.
    alpha controls CNN weight: final_prob = alpha*cnn + (1-alpha)*tabular
    """
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    fold_metrics = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['label']), 1):
        train_df = df.iloc[train_idx]
        val_df   = df.iloc[val_idx]
        
        # ── CNN probs (inference only, no training) ────────────────
        cnn_probs_val, labels_val = get_cnn_probabilities(
            val_df, CNN_CLASS, CNN_WEIGHTS, TRANSFORM, CNN_EMBED_DIM, device)
        
        # ── Tabular LR ────────────────────────────────────────────
        scaler = StandardScaler()
        X_train = scaler.fit_transform(train_df[feature_cols].values)
        X_val   = scaler.transform(val_df[feature_cols].values)
        y_train = train_df['label'].values
        
        lr = LogisticRegression(max_iter=1000, random_state=SEED)
        lr.fit(X_train, y_train)
        tab_probs_val = lr.predict_proba(X_val)[:, 1]
        
        # ── Combine ───────────────────────────────────────────────
        fused_probs = alpha * cnn_probs_val + (1 - alpha) * tab_probs_val
        
        m = compute_metrics(labels_val, fused_probs)
        m['fold'] = fold
        fold_metrics.append(m)
        print(f'  Fold {fold}: AUC={m["auc"]:.3f}  F1={m["f1"]:.3f}')
    
    return pd.DataFrame(fold_metrics)

In [ ]:
# Run late fusion for each feature group + all combined
late_fusion_results = {}

for group_name, feature_cols in FEATURE_GROUPS.items():
    available = [f for f in feature_cols if f in balanced.columns]
    if not available:
        print(f'[SKIP] {group_name} — columns not found in data')
        continue
    print(f'\nLate fusion — {group_name}: {available}')
    late_fusion_results[f'late_{group_name}'] = run_late_fusion_cv(balanced, available)

# All features combined
all_cols = [f for group in FEATURE_GROUPS.values() for f in group if f in balanced.columns]
print(f'\nLate fusion — ALL features: {all_cols}')
late_fusion_results['late_ALL'] = run_late_fusion_cv(balanced, all_cols)

## 5. Feature fusion (your proposed approach)

Strip the CNN head → get a learned image embedding → concatenate with tabular features → train a small MLP fusion head.

The CNN backbone can be:
- **Frozen** (`FREEZE_CNN=True`): only the fusion head trains. Faster, avoids overfitting on small data. Good first run.
- **Unfrozen** (`FREEZE_CNN=False`): full end-to-end fine-tuning. Better if you have enough data.

In [ ]:
class MultimodalDataset(Dataset):
    """
    Wraps a MONAI transform pipeline and also returns tabular features.
    Each item: {'image_tensor': ..., 'tabular': ..., 'label': ...}
    """
    def __init__(self, df, feature_cols, transform):
        self.transform    = transform
        self.feature_cols = feature_cols
        self.paths  = df['path'].tolist()
        self.labels = df['label'].tolist()
        # Tabular as float32 tensor — scaler applied externally
        self.tabular = torch.tensor(
            df[feature_cols].values.astype(np.float32))
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        # Apply MONAI image transform
        item = self.transform({'image': self.paths[idx], 'label': self.labels[idx]})
        return {
            'image':   item['image'],
            'tabular': self.tabular[idx],
            'label':   torch.tensor(self.labels[idx], dtype=torch.float32),
        }


class MultimodalFusionModel(nn.Module):
    """
    Y-shaped architecture:
      - CNN branch: pretrained backbone with head removed → image_embed_dim
      - Tabular branch: small MLP → tabular_hidden
      - Fusion head: concat → MLP → logit
    """
    def __init__(self, cnn_backbone, image_embed_dim, n_tabular,
                 tabular_hidden=16, dropout_rate=0.3, freeze_cnn=True):
        super().__init__()
        self.cnn = cnn_backbone
        
        if freeze_cnn:
            for p in self.cnn.parameters():
                p.requires_grad = False
        
        self.tabular_branch = nn.Sequential(
            nn.Linear(n_tabular, tabular_hidden),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(tabular_hidden, tabular_hidden),
            nn.ReLU(),
        )
        fused_dim = image_embed_dim + tabular_hidden
        self.head = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(fused_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )
    
    def forward(self, image, tabular):
        img_emb = self.cnn(image)              # (B, image_embed_dim)
        tab_emb = self.tabular_branch(tabular) # (B, tabular_hidden)
        fused   = torch.cat([img_emb, tab_emb], dim=1)
        return self.head(fused)                # (B, 1)

In [ ]:
def build_cnn_backbone(model_class, weights_path, embed_dim, device):
    """Load pretrained CNN and remove its classification head."""
    model = model_class(dropout_rate=DROPOUT)
    model.load_state_dict(torch.load(weights_path, map_location=device))
    if hasattr(model, 'fc2'):   # custom 2D/3D CNNs — embed_dim = 32
        model.fc2 = nn.Identity()
    elif hasattr(model, 'fc'):  # 25d_resnet — embed_dim = 512
        model.fc = nn.Identity()
    return model.to(device)


def run_feature_fusion_cv(df, feature_cols, n_folds=FOLDS):
    """Train and evaluate the feature-fusion model with k-fold CV."""
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    fold_metrics = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['label']), 1):
        train_df = df.iloc[train_idx].reset_index(drop=True)
        val_df   = df.iloc[val_idx].reset_index(drop=True)
        
        # Scale tabular features (fit on train, apply to val)
        scaler = StandardScaler()
        train_df = train_df.copy()
        val_df   = val_df.copy()
        train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols].values)
        val_df[feature_cols]   = scaler.transform(val_df[feature_cols].values)
        
        # Datasets
        train_ds = MultimodalDataset(train_df, feature_cols, TRANSFORM)
        val_ds   = MultimodalDataset(val_df,   feature_cols, TRANSFORM)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
        
        # Build fresh model for each fold
        backbone = build_cnn_backbone(CNN_CLASS, CNN_WEIGHTS, CNN_EMBED_DIM, device)
        model = MultimodalFusionModel(
            cnn_backbone    = backbone,
            image_embed_dim = CNN_EMBED_DIM,
            n_tabular       = len(feature_cols),
            tabular_hidden  = 16,
            dropout_rate    = DROPOUT,
            freeze_cnn      = FREEZE_CNN,
        ).to(device)
        
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
        
        best_auc, best_probs, best_labels = 0, None, None
        
        for epoch in range(EPOCHS):
            # Train
            model.train()
            for batch in train_loader:
                imgs    = batch['image'].to(device)
                tabular = batch['tabular'].to(device)
                labels  = batch['label'].to(device).view(-1, 1)
                optimizer.zero_grad()
                loss = criterion(model(imgs, tabular), labels)
                loss.backward()
                optimizer.step()
            
            # Validate
            model.eval()
            all_probs, all_labels = [], []
            with torch.no_grad():
                for batch in val_loader:
                    imgs    = batch['image'].to(device)
                    tabular = batch['tabular'].to(device)
                    labels  = batch['label'].numpy().flatten()
                    logits  = model(imgs, tabular).cpu().numpy().flatten()
                    probs   = 1 / (1 + np.exp(-logits))
                    all_probs.extend(probs)
                    all_labels.extend(labels)
            
            try:
                auc = roc_auc_score(all_labels, all_probs)
            except ValueError:
                auc = 0.5
            
            if auc > best_auc:
                best_auc    = auc
                best_probs  = np.array(all_probs)
                best_labels = np.array(all_labels)
        
        m = compute_metrics(best_labels, best_probs)
        m['fold'] = fold
        fold_metrics.append(m)
        print(f'  Fold {fold}: AUC={m["auc"]:.3f}  F1={m["f1"]:.3f}')
    
    return pd.DataFrame(fold_metrics)

In [ ]:
feature_fusion_results = {}

for group_name, feature_cols in FEATURE_GROUPS.items():
    available = [f for f in feature_cols if f in balanced.columns]
    if not available:
        print(f'[SKIP] {group_name} — columns not found')
        continue
    print(f'\nFeature fusion — {group_name}: {available}')
    feature_fusion_results[f'feature_{group_name}'] = run_feature_fusion_cv(balanced, available)

# All features combined
print(f'\nFeature fusion — ALL features: {all_cols}')
feature_fusion_results['feature_ALL'] = run_feature_fusion_cv(balanced, all_cols)

## 6. Information gain table

Compile all results into one table. This is the key output for your thesis.

In [ ]:
all_results = {**late_fusion_results, **feature_fusion_results}

rows = []
for name, df_m in all_results.items():
    fusion_type, group = name.split('_', 1)
    for metric in ['auc', 'f1', 'accuracy', 'recall', 'precision']:
        rows.append({
            'model':    name,
            'fusion':   fusion_type,
            'features': group,
            'metric':   metric,
            'mean':     df_m[metric].mean(),
            'std':      df_m[metric].std(),
        })

results_long = pd.DataFrame(rows)

# Pivot to wide format for easy reading
summary = results_long.pivot_table(
    index=['fusion', 'features'], columns='metric', values=['mean', 'std']
).round(3)

print('\n── Information gain table (mean ± std across folds) ─────────────')
print(summary.to_string())

# Save to CSV
summary.to_csv('multimodal_results.csv')
print('\nSaved to multimodal_results.csv')

## 7. Boxplots

In [ ]:
METRICS_TO_PLOT = ['auc', 'f1']

COLORS = {
    'late':    '#4C72B0',   # blue — late fusion
    'feature': '#55A868',   # green — feature fusion
}

# Sort models by median AUC
model_order = (
    results_long[results_long['metric'] == 'auc']
    .groupby('model')['mean']
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, axes = plt.subplots(1, len(METRICS_TO_PLOT),
                         figsize=(len(model_order) * 1.6 * len(METRICS_TO_PLOT), 5))

for ax, metric in zip(axes, METRICS_TO_PLOT):
    data, labels, colors = [], [], []
    for m in model_order:
        fold_df = all_results[m]
        data.append(fold_df[metric].values)
        labels.append(m.replace('_', '\n'))
        colors.append(COLORS[m.split('_')[0]])
    
    bp = ax.boxplot(
        data, patch_artist=True, labels=labels,
        medianprops=dict(color='white', linewidth=2),
        widths=0.5,
    )
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.82)
    
    for i, d in enumerate(data):
        ax.text(i+1, np.median(d)+0.003, f'{np.median(d):.3f}',
                ha='center', va='bottom', fontsize=7)
    
    ax.set_title(metric.upper(), fontsize=11)
    ax.set_ylim(max(0, min(np.concatenate(data)) - 0.05), 1.05)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='x', labelsize=8)

legend_handles = [
    mpatches.Patch(color=COLORS['late'],    label='Late fusion (CNN prob + tabular LR)'),
    mpatches.Patch(color=COLORS['feature'], label='Feature fusion (CNN embed + tabular MLP)'),
]
axes[-1].legend(handles=legend_handles, fontsize=8, loc='lower right')
fig.suptitle('Multimodal fusion — information gain by feature group', fontsize=13, y=1.01)
fig.tight_layout()
plt.savefig('multimodal_boxplots.png', dpi=160, bbox_inches='tight')
plt.show()
print('Saved multimodal_boxplots.png')